# C2 - StudentLife INSPECTION notebook (step 1)

**Purpose:** only LOOK at the real StudentLife files. Nothing is cleaned, deleted, merged or trained here.

What this notebook answers:
1. Which folders/files exist and how many participants each has
2. What columns each table really has (no invented names)
3. Time coverage of each sensing source
4. Which participants have which sources (overlap)
5. What targets exist (EMA stress/sleep, surveys) and how many answers per participant

**Status:** exploration only. Everything here is preliminary, not validated.
StudentLife is NOT mixed with SWELL-KW.

## SCOPE LIMIT - only sources that serve C2 inputs are inspected
| Used? | StudentLife source | Why |
|---|---|---|
| YES | `sensing/phonelock`, `sensing/dark`, `sensing/phonecharge` | screen use + sleep / recent-inactivity proxies |
| YES | `sensing/activity` | physical-activity context |
| YES | `app_usage` | phone/app usage (header + timestamps only) |
| YES (reference only) | `EMA/response/Sleep`, `EMA/response/Stress` | candidate TARGETS / sleep reference, never model inputs |
| YES (reference only) | `survey/PerceivedStressScale`, `psqi`, `panas` | stress / sleep quality / affect reference |
| NO | `gps`, `audio`, `conversation`, `bluetooth`, `wifi`, `wifi_location` | not a C2 input area |
| NO | `calendar`, `education` (deadlines, grades, class) | deadline/schedule pressure belongs to C1 |
| NO | `call_log`, `sms`, `dinning`, other EMA categories | not a C2 input area |
| NO | `PHQ-9`, `BigFive`, `Loneliness`, `Flourishing`, `vr_12` | clinical/personality/social scales: outside C2 scope and could drift into medical claims |

If you want to add a dropped source later, that is a decision for you to approve, not a default.

## Cell 1 - Connect Google Drive and set the dataset path
Edit `DATA_ROOT` so it points to the folder that contains `app_usage`, `EMA`, `sensing`, `survey`.
Tip: you only need to upload the in-scope folders (see the SCOPE LIMIT table above) - this saves most of the ~3 GB.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

import os

def find_data_root(start='/content/drive/MyDrive', max_depth=10):
    """Search Drive for the folder that contains sensing/phonelock (the StudentLife root)."""
    start_depth = start.rstrip('/').count('/')
    for dirpath, dirnames, _ in os.walk(start):
        if dirpath.count('/') - start_depth >= max_depth:
            dirnames[:] = []          # do not go deeper
            continue
        if os.path.basename(dirpath) == 'sensing' and 'phonelock' in dirnames:
            return Path(dirpath).parent
    return None

# Automatic search. If it fails, type the path yourself on the next line instead.
DATA_ROOT = find_data_root()
# DATA_ROOT = Path('/content/drive/MyDrive/<your folder>/StudentLife_dataset')

# Where summary tables from this notebook are saved (you can copy them to the repo's outputs/ later)
OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_ROOT is not None and DATA_ROOT.exists(), \
    'StudentLife folder not found automatically - set DATA_ROOT by hand (see comment above).'
print('OK, dataset folder found:', DATA_ROOT)

## Cell 2 - Folder overview
Counts files per folder. Participant ids look like `u00`, `u01`, ... and appear inside file names.

In [ ]:
import re
import pandas as pd

def uid_from_name(name):
    """Extract participant id like 'u07' from a file name; None if absent."""
    m = re.search(r'(u\d{2})', name)
    return m.group(1) if m else None

# ONLY these folders are inspected (C2 scope). Everything else in StudentLife is ignored.
C2_FOLDERS = [
    'sensing/phonelock', 'sensing/dark', 'sensing/phonecharge', 'sensing/activity',
    'app_usage', 'EMA/response/Sleep', 'EMA/response/Stress', 'survey',
]
C2_SURVEYS = ['PerceivedStressScale.csv', 'psqi.csv', 'panas.csv']

rows = []
for rel in C2_FOLDERS:
    folder = DATA_ROOT / rel
    if rel == 'survey':
        files = [folder / s for s in C2_SURVEYS if (folder / s).exists()]
    else:
        files = [f for f in folder.rglob('*') if f.is_file()]
    size_mb = sum(f.stat().st_size for f in files) / 1e6
    uids = {uid_from_name(f.name) for f in files} - {None}
    rows.append({'folder': rel, 'n_files': len(files),
                 'size_MB': round(size_mb, 1), 'n_participants': len(uids)})

overview = pd.DataFrame(rows)
display(overview)
overview.to_csv(OUT_DIR / 'studentlife_folder_overview.csv', index=False)

## Cell 3 - Participant coverage per source
A matrix of which participants have data in which source. Missing participants = missing data we must handle later.

In [ ]:
sensing_dir = DATA_ROOT / 'sensing'
ema_resp = DATA_ROOT / 'EMA' / 'response'
sources = {
    'app_usage':           DATA_ROOT / 'app_usage',
    'sensing/phonelock':   sensing_dir / 'phonelock',
    'sensing/dark':        sensing_dir / 'dark',
    'sensing/phonecharge': sensing_dir / 'phonecharge',
    'sensing/activity':    sensing_dir / 'activity',
    'EMA/Sleep':           ema_resp / 'Sleep',
    'EMA/Stress':          ema_resp / 'Stress',
}

cover = {}
for name, folder in sources.items():
    cover[name] = {uid_from_name(f.name) for f in folder.iterdir() if f.is_file()} - {None}

all_uids = sorted(set().union(*cover.values()))
coverage = pd.DataFrame({name: [u in s for u in all_uids] for name, s in cover.items()}, index=all_uids)
print('Participants seen anywhere:', len(all_uids))
display(coverage.sum().rename('n_participants').to_frame())
coverage.to_csv(OUT_DIR / 'studentlife_participant_coverage.csv')

## Cell 4 - Real columns of every CSV table
Reads only the first rows of ONE participant file per source, so it is fast. These are the true column names we may use later.

In [ ]:
def peek_csv(path, n=3):
    """Read the first n rows only (fast, avoids loading big files)."""
    return pd.read_csv(path, nrows=n, on_bad_lines='skip')

csv_sources = {k: v for k, v in sources.items() if not k.startswith('EMA/')}
column_rows = []
for name, folder in csv_sources.items():
    files = sorted(folder.glob('*.csv'))
    if not files:
        continue
    sample = peek_csv(files[0])
    print(f'\n=== {name}  (example file: {files[0].name})')
    print('columns:', list(sample.columns))
    display(sample)
    column_rows.append({'source': name, 'example_file': files[0].name,
                        'columns': ' | '.join(map(str, sample.columns))})

pd.DataFrame(column_rows).to_csv(OUT_DIR / 'studentlife_columns.csv', index=False)

## Cell 5 - Time coverage of the sensing sources
Timestamps in StudentLife are Unix seconds (we confirm below). Needed to see how many days each participant has.
Uses the small interval/inference files only (`activity`, `phonelock`, `dark`, `phonecharge`).

In [ ]:
def time_range(path):
    """Return (n_rows, first_time, last_time) in seconds using the first/last time-like column."""
    df = pd.read_csv(path, on_bad_lines='skip')
    time_col = 'timestamp' if 'timestamp' in df.columns else 'start'
    end_col = 'end' if 'end' in df.columns else time_col
    return len(df), df[time_col].min(), df[end_col].max()

rows = []
for sub in ['activity', 'phonelock', 'dark', 'phonecharge']:
    for f in sorted((sensing_dir / sub).glob('*.csv')):
        try:
            n, t0, t1 = time_range(f)
            rows.append({'source': sub, 'uid': uid_from_name(f.name), 'n_rows': n,
                         'first': pd.to_datetime(t0, unit='s'),
                         'last': pd.to_datetime(t1, unit='s')})
        except Exception as e:
            print('Could not read', f.name, '->', e)

tr = pd.DataFrame(rows)
tr['days_covered'] = (tr['last'] - tr['first']).dt.days
display(tr.groupby('source')[['n_rows', 'days_covered']].describe().T)
tr.to_csv(OUT_DIR / 'studentlife_sensing_time_ranges.csv', index=False)

## Cell 6 - EMA (in-the-moment answers) structure
EMA files are JSON. We check the real keys, how many responses each participant gave, and the answer values.
These are SELF-REPORT answers: if used, they would be TARGETS / reference labels, not model inputs.

In [ ]:
import json

def load_ema(category):
    """Load all participants' JSON for one EMA category into one DataFrame (adds uid)."""
    frames = []
    for f in sorted((ema_resp / category).glob('*.json')):
        with open(f) as fh:
            data = json.load(fh)
        if not data:
            continue
        df = pd.DataFrame(data)
        df['uid'] = uid_from_name(f.name)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

ema = {}
for cat in ('Stress', 'Sleep'):
    ema[cat] = load_ema(cat)
    df = ema[cat]
    print(f'\n=== EMA {cat}: {len(df)} responses, {df["uid"].nunique()} participants')
    print('columns:', list(df.columns))
    display(df.head())
    for col in df.columns:
        if col not in ('uid', 'resp_time', 'location', 'null') and df[col].nunique() < 25:
            print(f'\nvalue counts for "{col}":')
            print(df[col].value_counts(dropna=False))
    per_user = df.groupby('uid').size()
    print('\nresponses per participant (min / median / max):',
          per_user.min(), per_user.median(), per_user.max())

## Cell 7 - EMA question definitions (what each answer code means)
Read from `EMA_definition.json` so we never guess the meaning of a code.

In [ ]:
with open(DATA_ROOT / 'EMA' / 'EMA_definition.json') as fh:
    ema_def = json.load(fh)

for item in ema_def:
    if item.get('name') in ('Stress', 'Sleep'):
        print('\n###', item['name'])
        for q in item['questions']:
            print(' -', q.get('question_id'), '|', q.get('question_text'))
            print('     options:', q.get('options'))

## Cell 8 - Survey tables (pre/post questionnaires)
Each survey has one row per participant per `type` (pre / post). Checks shape, missing values and columns.

In [ ]:
survey_dir = DATA_ROOT / 'survey'
for name in C2_SURVEYS:   # only the 3 in-scope surveys (PSS, PSQI, PANAS)
    f = survey_dir / name
    df = pd.read_csv(f)
    print(f'\n=== {f.name}: {df.shape[0]} rows x {df.shape[1]} columns, '
          f'participants={df["uid"].nunique()}')
    print('rows per type:', df['type'].value_counts().to_dict())
    missing = df.isna().sum()
    print('columns with missing values:', int((missing > 0).sum()), 'of', df.shape[1])

## Note on task difficulty
StudentLife has no clean task-difficulty field. `education/deadlines` is deadline pressure (C1 territory), so it is deliberately NOT inspected here. Task difficulty will come from our own Part B / pre-post data.

## Cell 9 - What to send back to Claude
Copy/paste (or screenshot) the printed outputs of Cells 2, 3, 5, 6, 7 and 8. The CSV summaries were also saved in `OUT_DIR`.
Next step (after we check these results together): decide the preliminary target and feature mapping.